# 🧠 Agent Memory System

A practical GenAI project demonstrating **short-term conversation memory, long-term memory, semantic retrieval, user preferences, memory extraction, consolidation, updates, and memory-aware responses**.

**Evolution:** Short-Term Memory → Long-Term Memory → Semantic Retrieval → Memory Consolidation → LangGraph/MCP Production Memory.

## 🏗️ Architecture

```text
User → Conversation → Memory Extractor
                         ├─ Facts
                         ├─ Preferences
                         ├─ Goals / Projects
                         ↓
                    Memory Store
                         ↓
                   Semantic Search
                         ↓
                  Relevant Memories
                         ↓
                    LLM Agent
                         ↓
                 Memory-aware Response
```

## 1. Setup

Install dependencies:

```python
%pip install -q openai python-dotenv pydantic numpy
```

Create `.env`:

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

In [ ]:
%pip install -q openai python-dotenv pydantic numpy

In [ ]:
import os, json, uuid
from datetime import datetime, timezone
from typing import Optional, List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

client = None
if OPENROUTER_API_KEY:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
    print(f"✅ LLM configured: {MODEL}")
else:
    print("⚠️ Add OPENROUTER_API_KEY to .env to enable LLM calls.")

## 2. Memory Data Models

In [ ]:
class Memory(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    user_id: str
    content: str
    memory_type: str = "fact"
    importance: float = 0.5
    created_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    updated_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    metadata: Dict[str, Any] = Field(default_factory=dict)

class Message(BaseModel):
    role: str
    content: str
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

## 3. Short-Term Conversation Memory

In [ ]:
class ConversationMemory:
    def __init__(self, max_messages=10):
        self.max_messages = max_messages
        self.messages = []

    def add(self, role, content):
        self.messages.append(Message(role=role, content=content))
        self.messages = self.messages[-self.max_messages:]

    def get_recent(self):
        return list(self.messages)

    def clear(self):
        self.messages.clear()

    def as_text(self):
        return "\n".join(f"{m.role.upper()}: {m.content}" for m in self.messages)

short_term_memory = ConversationMemory(max_messages=6)
short_term_memory.add("user", "I am building an AI interview assistant.")
short_term_memory.add("assistant", "What technology stack are you using?")
short_term_memory.add("user", "I prefer Python, React, and MongoDB.")
print(short_term_memory.as_text())

## 4. Long-Term Memory Store

In [ ]:
class MemoryStore:
    def __init__(self):
        self.memories = {}

    def add(self, memory):
        self.memories[memory.id] = memory
        return memory

    def get(self, memory_id):
        return self.memories.get(memory_id)

    def get_user_memories(self, user_id):
        return [m for m in self.memories.values() if m.user_id == user_id]

    def delete(self, memory_id):
        return self.memories.pop(memory_id, None) is not None

    def update(self, memory_id, content, importance=None):
        memory = self.get(memory_id)
        if not memory:
            return None
        memory.content = content
        if importance is not None:
            memory.importance = importance
        memory.updated_at = datetime.now(timezone.utc).isoformat()
        return memory

memory_store = MemoryStore()

## 5. LLM Helper

In [ ]:
def ask_llm(system_prompt, user_prompt, temperature=0.2):
    if not client:
        raise RuntimeError("OPENROUTER_API_KEY is not configured.")
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content

## 6. Automatic Memory Extraction

In [ ]:
def extract_memories(user_id, conversation):
    system = """You are a memory extraction component for an AI assistant.
Extract only useful long-term information explicitly stated by the user:
stable preferences, facts, goals, ongoing projects, and recurring constraints.
Do NOT store passwords, API keys, financial credentials, highly sensitive information,
temporary details, or inferred information.

Return ONLY valid JSON:
{"memories":[{"content":"...","memory_type":"preference|fact|goal|project","importance":0.0}]}
If nothing should be remembered: {"memories":[]}"""

    raw = ask_llm(system, conversation, temperature=0.0)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return []

    return [
        Memory(
            user_id=user_id,
            content=item["content"],
            memory_type=item.get("memory_type", "fact"),
            importance=float(item.get("importance", 0.5)),
        )
        for item in data.get("memories", [])
    ]

# Example:
# memories = extract_memories("demo-user", "I prefer Python and I am building an AI interview assistant.")
# for memory in memories:
#     memory_store.add(memory)

## 7. Lightweight Semantic Retrieval

In [ ]:
def tokenize(text):
    return {x.lower().strip(".,!?;:()[]{}") for x in text.split() if x.strip()}

def similarity(query, document):
    q, d = tokenize(query), tokenize(document)
    if not q or not d:
        return 0.0
    return len(q & d) / len(q | d)

def retrieve_memories(user_id, query, top_k=5):
    memories = memory_store.get_user_memories(user_id)
    return sorted(
        memories,
        key=lambda m: similarity(query, m.content) + 0.15 * m.importance,
        reverse=True,
    )[:top_k]

## 8. Memory-Aware Agent

In [ ]:
def build_memory_context(user_id, query, top_k=5):
    memories = retrieve_memories(user_id, query, top_k)
    if not memories:
        return "No relevant long-term memories found."
    return "\n".join(f"- ({m.memory_type}) {m.content}" for m in memories)

def memory_aware_agent(user_id, user_message, conversation_memory):
    long_term = build_memory_context(user_id, user_message)
    recent = conversation_memory.as_text()

    system = """You are a memory-aware AI assistant.
Use memory only when relevant. Prefer recent conversation when it conflicts
with older memory. Do not expose internal memory records unnecessarily.
Do not claim to remember information that is not present."""

    prompt = f"""LONG-TERM MEMORY:
{long_term}

RECENT CONVERSATION:
{recent}

CURRENT USER MESSAGE:
{user_message}"""

    response = ask_llm(system, prompt, temperature=0.3)
    conversation_memory.add("user", user_message)
    conversation_memory.add("assistant", response)
    return response

# Example:
# conversation = ConversationMemory()
# print(memory_aware_agent("demo-user", "What stack do I prefer?", conversation))

## 9. Memory Consolidation

In [ ]:
def consolidate_memory(user_id, new_memory):
    existing = memory_store.get_user_memories(user_id)
    for old in existing:
        if similarity(new_memory.content, old.content) >= 0.7:
            old.content = new_memory.content
            old.importance = max(old.importance, new_memory.importance)
            old.updated_at = datetime.now(timezone.utc).isoformat()
            return old
    return memory_store.add(new_memory)

def add_memory_safely(user_id, content, memory_type="fact", importance=0.5):
    return consolidate_memory(
        user_id,
        Memory(user_id=user_id, content=content, memory_type=memory_type, importance=importance)
    )

## 10. Explicit Memory Management

In [ ]:
def list_memories(user_id):
    return memory_store.get_user_memories(user_id)

def forget_memory(memory_id):
    return memory_store.delete(memory_id)

def update_memory(memory_id, new_content):
    return memory_store.update(memory_id, new_content)

# Example:
# memory = add_memory_safely(
#     "demo-user",
#     "The user is working on an AI interview assistant.",
#     "project",
#     0.9
# )
# print(memory.model_dump())

## 11. Production Architecture

```text
React / Next.js
      ↓
FastAPI
      ↓
LangGraph Agent
      ├── Working Memory → Redis / graph state
      ├── Episodic Memory → PostgreSQL
      ├── Semantic Memory → pgvector / Qdrant
      ├── Procedural Memory → prompts / policies
      └── Tools / MCP
               ↓
              LLM
```

### Advanced extensions
- 🔗 LangGraph stateful memory
- 📚 Embedding-based semantic retrieval
- 🧠 Memory reflection and summarization
- 🕒 Memory decay / recency scoring
- 🎯 Importance and confidence scoring
- 🔐 User-controlled memory inspection and deletion
- 🔌 MCP memory tools
- 🧪 Memory retrieval evaluation
- 📊 Observability and token/cost tracking

## 🧪 Suggested Experiments

1. **Preference memory:** store a preferred programming language and retrieve it later.
2. **Project memory:** store an ongoing project and ask about it later.
3. **Memory update:** replace an old technology preference with a new one.
4. **Irrelevant memory:** add unrelated facts and verify retrieval relevance.
5. **Privacy:** provide an API-key-like string and verify the extractor refuses to store it.
6. **Contradiction:** store conflicting preferences and implement recency/confidence resolution.

## 🎯 Final Goal

Build an agent that can decide:

**What should I remember? → How important is it? → Where should it be stored? → When should I retrieve it? → Should it be updated or forgotten?**

That is the foundation of a production-grade **Agent Memory System**.